In [ ]:
"""
Credit Card Fraud Risk Management Project (Production-style)

Goal
- Build a fraud risk model to estimate P(fraud) per transaction (PD-like score)
- Provide model evaluation (ROC-AUC, PR-AUC, KS), calibration, and threshold strategy
- Produce artifacts that are useful for risk operations: score distribution, top drivers, decision thresholds

Notes
- This script is written to be readable and professional for a portfolio / interview.
- It includes defensive checks and clear separation between data, features, modeling, and evaluation.
"""

from __future__ import annotations

import os
import warnings
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    confusion_matrix,
)

warnings.filterwarnings("ignore")


# ----------------------------
# 0) Config
# ----------------------------
@dataclass
class Config:
    data_path: str = "张馨元/CreditCardFraud/CreditCardFraud_updated.csv" 
    target_col: str = "isFraud"
    time_col: str = "transactionDateTime"

    # Columns I dropped in the notebook as "mostly empty" or low value for modeling in this dataset version
    drop_cols: Tuple[str, ...] = (
        "Unnamed: 0",
        "echoBuffer",
        "merchantCity",
        "merchantState",
        "merchantZip",
        "posOnPremises",
        "recurringAuthInd",
    )

    # Datetime columns used to engineer tenure / recency / expiry signals
    datetime_cols: Tuple[str, ...] = (
        "transactionDateTime",
        "accountOpenDate",
        "dateOfLastAddressChange",
        "currentExpDate",
    )

    # For risk modeling, time split is recommended to mimic real deployment.
    # If you cannot do strict time split, fallback to random split.
    use_time_split: bool = True
    test_size: float = 0.2
    random_state: int = 42

    # Imbalance handling
    class_weight: str = "balanced"  # or None

    # Calibration: helps turn model scores into better probability estimates
    do_calibration: bool = True
    calibration_method: str = "isotonic"  # "sigmoid" or "isotonic"


cfg = Config()


# ----------------------------
# 1) Utilities (risk metrics)
# ----------------------------
def ks_statistic(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """
    KS = max_t |F_pos(t) - F_neg(t)|
    where F_pos is CDF of scores for positives and F_neg for negatives.
    Commonly used in credit risk / fraud scoring.
    """
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)

    pos_scores = y_score[y_true == 1]
    neg_scores = y_score[y_true == 0]

    if len(pos_scores) == 0 or len(neg_scores) == 0:
        return np.nan

    # Sort scores and compute empirical CDFs on the combined grid
    grid = np.sort(np.unique(y_score))
    pos_cdf = np.searchsorted(np.sort(pos_scores), grid, side="right") / len(pos_scores)
    neg_cdf = np.searchsorted(np.sort(neg_scores), grid, side="right") / len(neg_scores)
    return float(np.max(np.abs(pos_cdf - neg_cdf)))


def pick_threshold_by_precision(
    y_true: np.ndarray,
    y_score: np.ndarray,
    min_precision: float = 0.80,
) -> Dict[str, float]:
    """
    In fraud ops, we often want a threshold that meets a minimum precision
    to control false-positive cost (manual review load).
    """
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    # precision_recall_curve returns precision/recall for thresholds plus one extra point
    # thresholds length = len(precision)-1
    precision = precision[:-1]
    recall = recall[:-1]

    feasible = np.where(precision >= min_precision)[0]
    if len(feasible) == 0:
        # If cannot meet min precision, fallback to threshold maximizing F1
        f1 = 2 * precision * recall / (precision + recall + 1e-12)
        best_idx = int(np.nanargmax(f1))
        return {
            "threshold": float(thresholds[best_idx]),
            "precision": float(precision[best_idx]),
            "recall": float(recall[best_idx]),
            "policy": "fallback_max_f1",
        }

    # Among feasible thresholds, pick the one with highest recall (catch more fraud)
    best_idx = int(feasible[np.argmax(recall[feasible])])
    return {
        "threshold": float(thresholds[best_idx]),
        "precision": float(precision[best_idx]),
        "recall": float(recall[best_idx]),
        "policy": f"min_precision_{min_precision:.2f}",
    }


# ----------------------------
# 2) Data Loading & Cleaning
# ----------------------------
def load_data(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Cannot find data file at: {path}\n"
            f"Please place the CSV next to this script or update Config.data_path."
        )
    df = pd.read_csv(path)
    return df


def clean_and_prepare(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    df = df.copy()

    # Drop known low-value columns
    drop_existing = [c for c in cfg.drop_cols if c in df.columns]
    if drop_existing:
        df = df.drop(columns=drop_existing)

    # Basic target checks
    if cfg.target_col not in df.columns:
        raise ValueError(f"Target column '{cfg.target_col}' not found in data.")

    # Fill missing values for specific columns you handled in the notebook (if they exist)
    # Keep the logic explicit because it is meaningful for fraud data.
    if "acqCountry" in df.columns:
        df["acqCountry"] = df["acqCountry"].fillna("UNKNOWN")
    if "merchantCountryCode" in df.columns:
        df["merchantCountryCode"] = df["merchantCountryCode"].fillna("UNKNOWN")
    if "posEntryMode" in df.columns:
        df["posEntryMode"] = df["posEntryMode"].fillna(-1)
    if "posConditionCode" in df.columns:
        df["posConditionCode"] = df["posConditionCode"].fillna(-1)
    if "transactionType" in df.columns:
        df["transactionType"] = df["transactionType"].fillna("UNKNOWN")

    # Parse datetimes and create risk-relevant tenure / recency / expiry features
    df = add_time_features(df, cfg)

    # Add fraud-specific engineered features from your notebook + some common ones
    df = add_fraud_features(df)

    # Log transform amount to stabilize variance (as you did)
    if "transactionAmount" in df.columns:
        df["log_transactionAmount"] = np.log1p(df["transactionAmount"])

    return df


def add_time_features(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    df = df.copy()

    # Parse each datetime column if present
    # Different formats exist; we keep "errors=coerce" so bad strings become NaT.
    if "currentExpDate" in df.columns:
        df["currentExpDate"] = pd.to_datetime(df["currentExpDate"], format="%m/%Y", errors="coerce")
    if "accountOpenDate" in df.columns:
        df["accountOpenDate"] = pd.to_datetime(df["accountOpenDate"], format="%Y-%m-%d", errors="coerce")
    if "dateOfLastAddressChange" in df.columns:
        df["dateOfLastAddressChange"] = pd.to_datetime(df["dateOfLastAddressChange"], format="%Y-%m-%d", errors="coerce")
    if cfg.time_col in df.columns:
        df[cfg.time_col] = pd.to_datetime(df[cfg.time_col], format="%Y-%m-%dT%H:%M:%S", errors="coerce")

    # Feature engineering: days since events
    if cfg.time_col in df.columns and "accountOpenDate" in df.columns:
        df["days_since_account_open"] = (df[cfg.time_col] - df["accountOpenDate"]).dt.days

    if cfg.time_col in df.columns and "dateOfLastAddressChange" in df.columns:
        df["days_since_address_change"] = (df[cfg.time_col] - df["dateOfLastAddressChange"]).dt.days

    if cfg.time_col in df.columns and "currentExpDate" in df.columns:
        df["days_until_expiry"] = (df["currentExpDate"] - df[cfg.time_col]).dt.days
        df["is_card_expired"] = (df["currentExpDate"] < df[cfg.time_col]).astype(int)

    # Additional time-of-day / day-of-week patterns are common in fraud
    if cfg.time_col in df.columns:
        df["txn_hour"] = df[cfg.time_col].dt.hour
        df["txn_dayofweek"] = df[cfg.time_col].dt.dayofweek  # Mon=0
        df["is_weekend"] = df["txn_dayofweek"].isin([5, 6]).astype(int)

    return df


def add_fraud_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # CVV match (your notebook)
    if "cardCVV" in df.columns and "enteredCVV" in df.columns:
        df["cvv_match"] = (df["cardCVV"] == df["enteredCVV"]).astype(int)

    # Card last4 frequency in dataset (your notebook)
    if "cardLast4Digits" in df.columns:
        card4_freq = df["cardLast4Digits"].value_counts(dropna=False)
        df["card4_freq"] = df["cardLast4Digits"].map(card4_freq)

    # A common fraud signal: high utilization
    if "creditLimit" in df.columns and "currentBalance" in df.columns:
        df["utilization"] = df["currentBalance"] / (df["creditLimit"] + 1e-9)

    # Another signal: available money ratio
    if "availableMoney" in df.columns and "creditLimit" in df.columns:
        df["available_ratio"] = df["availableMoney"] / (df["creditLimit"] + 1e-9)

    return df


# ----------------------------
# 3) Train/Test Split
# ----------------------------
def time_based_split(df: pd.DataFrame, cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Sort by transaction time and split by the last X% as test.
    This reduces leakage and better simulates real-world deployment.
    """
    if cfg.time_col not in df.columns:
        raise ValueError(f"Time column '{cfg.time_col}' not found for time split.")

    df_sorted = df.sort_values(cfg.time_col)
    cut = int(len(df_sorted) * (1 - cfg.test_size))
    train_df = df_sorted.iloc[:cut].copy()
    test_df = df_sorted.iloc[cut:].copy()
    return train_df, test_df


# ----------------------------
# 4) Modeling Pipeline
# ----------------------------
def build_pipeline(X: pd.DataFrame, cfg: Config) -> Pipeline:
    """
    Production-like pipeline:
    - Identify numeric and categorical columns
    - Impute missing values
    - One-hot encode categorical variables
    - Scale numeric variables
    - Train logistic regression with imbalance handling
    """
    # Exclude target and raw datetime columns from features
    # We keep engineered features (days_since_*, txn_hour, etc.)
    X = X.copy()

    numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

    # If you want to drop raw IDs, do it here (often recommended)
    # These IDs can leak / overfit if they are too specific.
    for raw_id in ["accountNumber", "customerId", "cardId", "merchantId"]:
        if raw_id in numeric_cols:
            numeric_cols.remove(raw_id)
        if raw_id in categorical_cols:
            categorical_cols.remove(raw_id)

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    base_model = LogisticRegression(
        max_iter=2000,
        n_jobs=None,
        class_weight=cfg.class_weight if cfg.class_weight != "none" else None,
    )

    clf = Pipeline(
        steps=[
            ("prep", preprocessor),
            ("model", base_model),
        ]
    )

    # Optional: probability calibration (important in risk management)
    if cfg.do_calibration:
        # Use CV calibration on training data
        clf = CalibratedClassifierCV(
            estimator=clf,
            method=cfg.calibration_method,
            cv=3,
        )

    return clf


# ----------------------------
# 5) Evaluation & Reporting
# ----------------------------
def evaluate_binary_classifier(
    y_true: np.ndarray,
    y_score: np.ndarray,
    threshold: float,
) -> Dict[str, float]:
    """
    Returns key risk metrics at a given threshold.
    """
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    precision = tp / (tp + fp + 1e-12)
    recall = tp / (tp + fn + 1e-12)
    fpr = fp / (fp + tn + 1e-12)

    return {
        "threshold": float(threshold),
        "precision": float(precision),
        "recall": float(recall),
        "false_positive_rate": float(fpr),
        "tp": float(tp),
        "fp": float(fp),
        "tn": float(tn),
        "fn": float(fn),
    }


def run_training(df: pd.DataFrame, cfg: Config) -> None:
    # Remove rows with missing target
    df = df.dropna(subset=[cfg.target_col]).copy()
    df[cfg.target_col] = df[cfg.target_col].astype(int)

    # Split
    if cfg.use_time_split and cfg.time_col in df.columns:
        train_df, test_df = time_based_split(df, cfg)
    else:
        train_df, test_df = train_test_split(
            df, test_size=cfg.test_size, random_state=cfg.random_state, stratify=df[cfg.target_col]
        )

    # Separate X/y
    feature_df_train = train_df.drop(columns=[cfg.target_col])
    feature_df_test = test_df.drop(columns=[cfg.target_col])

    y_train = train_df[cfg.target_col].values
    y_test = test_df[cfg.target_col].values

    # Build model pipeline
    model = build_pipeline(feature_df_train, cfg)

    # Fit
    model.fit(feature_df_train, y_train)

    # Score
    # CalibratedClassifierCV exposes predict_proba; Pipeline also does.
    y_score_test = model.predict_proba(feature_df_test)[:, 1]

    # Global metrics
    roc_auc = roc_auc_score(y_test, y_score_test)
    pr_auc = average_precision_score(y_test, y_score_test)
    ks = ks_statistic(y_test, y_score_test)

    print("\n=== Model Performance (Test) ===")
    print(f"ROC-AUC : {roc_auc:.4f}")
    print(f"PR-AUC  : {pr_auc:.4f}")
    print(f"KS      : {ks:.4f}")

    # Threshold policy: risk operations often set thresholds by precision to control cost
    policy = pick_threshold_by_precision(y_test, y_score_test, min_precision=0.80)
    print("\n=== Threshold Policy ===")
    print(policy)

    threshold_metrics = evaluate_binary_classifier(
        y_true=y_test,
        y_score=y_score_test,
        threshold=policy["threshold"],
    )
    print("\n=== Confusion Metrics at Policy Threshold ===")
    for k, v in threshold_metrics.items():
        if isinstance(v, float):
            print(f"{k:>20}: {v:.4f}")
        else:
            print(f"{k:>20}: {v}")

    # Score banding: useful for risk reporting (e.g., 10 buckets)
    print("\n=== Score Banding (10 buckets) ===")
    score_banding_report(y_test, y_score_test, n_bins=10)

    # Optional: save a scoring output for portfolio / stakeholder presentation
    scored = test_df[[cfg.target_col]].copy()
    scored["score_pd"] = y_score_test
    out_path = "scored_test_sample.csv"
    scored.to_csv(out_path, index=False)
    print(f"\nSaved scored output to: {out_path}")


def score_banding_report(y_true: np.ndarray, y_score: np.ndarray, n_bins: int = 10) -> None:
    """
    Create score bands and compute fraud rate per band.
    This is a common risk management artifact:
    - higher bands should have higher fraud rates if model is useful.
    """
    df = pd.DataFrame({"y": y_true, "score": y_score})
    df["band"] = pd.qcut(df["score"], q=n_bins, duplicates="drop")

    agg = df.groupby("band").agg(
        count=("y", "size"),
        fraud_rate=("y", "mean"),
        avg_score=("score", "mean"),
    )
    # show highest-risk band first
    agg = agg.sort_values("avg_score", ascending=False)

    # Pretty print
    with pd.option_context("display.max_rows", 50, "display.width", 120):
        print(agg.to_string())


# ----------------------------
# 6) Main
# ----------------------------
def main() -> None:
    df_raw = load_data(cfg.data_path)
    df = clean_and_prepare(df_raw, cfg)

    # Defensive: drop raw datetime columns to avoid accidental leakage,
    # since we already created engineered time features.
    for col in cfg.datetime_cols:
        if col in df.columns:
            df = df.drop(columns=[col])

    run_training(df, cfg)


if __name__ == "__main__":
    main()